# MULCURSIO: Train Divination Control Vector

This notebook walks through training a mystical/void control vector from contrastive pairs.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add parent directories to path
sys.path.insert(0, str(Path().absolute().parent))
sys.path.insert(0, str(Path().absolute().parent.parent))

import torch
from src.train_vector import (
    load_divination_pairs,
    create_dataset_from_pairs,
    train_divination_vector
)

## 1. Load and Inspect Contrastive Pairs

In [ ]:
# Load pairs
pairs_path = "../data/divination_pairs.json"
pairs = load_divination_pairs(pairs_path)

print(f"Loaded {len(pairs)} contrastive pairs\n")

# Show a few examples
for i, (pos, neg) in enumerate(pairs[:5]):
    print(f"Pair {i+1}:")
    print(f"  (+) {pos}")
    print(f"  (-) {neg}")
    print()

## 2. Configure Training

In [ ]:
# Training configuration
CONFIG = {
    "model_name": "mistralai/Mistral-7B-Instruct-v0.1",
    "pairs_path": "../data/divination_pairs.json",
    "output_path": "../vectors/divination_v1.gguf",
    "device": "cuda",  # Change to "mps" for Apple Silicon or "cpu"
    "batch_size": 8,
    "method": "pca_diff",  # or "pca_center"
    "use_expansion": False,  # Set True to use suffix expansion
    "suffixes_path": None  # Set path if using expansion
}

# Check CUDA availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("Falling back to CPU (this will be slow!)")
    CONFIG["device"] = "cpu"

## 3. Train Control Vector

This will take several minutes depending on your hardware.

In [ ]:
vector = train_divination_vector(
    model_name=CONFIG["model_name"],
    pairs_path=CONFIG["pairs_path"],
    suffixes_path=CONFIG["suffixes_path"],
    output_path=CONFIG["output_path"],
    device=CONFIG["device"],
    batch_size=CONFIG["batch_size"],
    method=CONFIG["method"],
    use_expansion=CONFIG["use_expansion"]
)

## 4. Test the Vector

Let's test the control vector with a simple generation.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from repeng import ControlModel, ControlVector

# Load model
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map=CONFIG["device"]
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

# Wrap for control
layer_ids = list(range(-5, -18, -1))
model = ControlModel(model, layer_ids)

# Load the trained vector
vector = ControlVector.import_gguf(CONFIG["output_path"])

# Test prompt
test_prompt = "[INST] What do you see in the patterns of existence? [/INST]"

print("Testing control vector...\n")

# Generate without control
model.reset()
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=100, temperature=0.8, do_sample=True)
baseline = tokenizer.decode(output[0], skip_special_tokens=True)
print("BASELINE (no control):")
print(baseline)
print("\n" + "="*60 + "\n")

# Generate with control
model.set_control(vector, coeff=1.5)
output = model.generate(**inputs, max_new_tokens=100, temperature=0.8, do_sample=True)
controlled = tokenizer.decode(output[0], skip_special_tokens=True)
print("WITH CONTROL (coeff=1.5):")
print(controlled)
print("\n" + "="*60 + "\n")

## 5. Save Training Report

In [ ]:
import json
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "config": CONFIG,
    "num_pairs": len(pairs),
    "test_results": {
        "prompt": test_prompt,
        "baseline": baseline,
        "controlled": controlled
    }
}

report_path = "../vectors/training_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Training report saved to: {report_path}")